# Capabilities M01–M13 (models, annotations, reduction, CRNT)

Prompts exactly as in the paper. Each prompt cell starts a fresh session with `new_chat()`; a multi-turn conversation continues in the cells below it without `new_chat()`. Where the paper's run answered a question from Dynalyst, the reply is in a cell after the prompt, as `reply('yes')`: it sends the reply only if Dynalyst has asked a question in this run, and otherwise does nothing.

To install Dynalyst, see `INSTALL.md` at the root of the repository. For what each notebook holds, see `README.md` in this folder.

## How to use
1. Run the **setup cell**, choose the model and paste its API key.
2. Run the **files cell**: it copies the extra models, the R01 time course and the papers from `model_files/` (next to this notebook) into `data/`.
3. Run any prompt cell. Prompts that use BioModels, PubMed or OmniPath need internet.
4. If a cell hangs, use *Kernel > Interrupt*; after a restart, run the setup and files cells again.

In [ ]:
import os, getpass
# Your API key, asked once and kept only in this session (never saved in the notebook).
# Leave it empty if NVIDIA_API_KEY is already set, or to type the key in the box of the setup cell.
if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA API key (nvapi-..., or Enter to skip): ").strip()

In [ ]:
import sys
from pathlib import Path

# The project folder: the first folder up from here that holds the dynalyst package.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dynalyst" / "__init__.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dynalyst import config
from dynalyst.env_check import check_env
from dynalyst.model_picker import pick_model
from dynalyst.notebook import ask, show_answer
from dynalyst.agents import unified_agent
from dynalyst.agents.unified_agent import reset_agents as new_chat   # fresh session per prompt


def ask_auto(query):
    """The prompt as the evaluation scripts sent it: no pause before AUTO-07p."""
    answer = unified_agent.ask(query, approval="auto")
    show_answer(answer)
    return answer

def reply(text="yes"):
    """Answer a question Dynalyst asked (which parameter, wait for a long run, a wider range).
    Sends nothing when no question is open, so a rerun that did not ask is not given a stray "yes"."""
    if not any(a.state["pending_approval"] for a in unified_agent._AGENTS.values()):
        print("No open question from Dynalyst in this run: nothing sent.")
        return None
    return ask(text)

check_env()
print("AUTO-07p:", config.AUTO_DIR)
pick_model();     # choose the model, give its API key if asked

In [ ]:
# Copy the extra model files, data and papers into the project (only files not already there)
import shutil
FILES = Path("model_files") if Path("model_files").exists() else ROOT / "notebooks" / "benchmarks" / "model_files"
DEST = {".xml": config.DATA_DIR / "sbml_models", ".ode": config.DATA_DIR / "sbml_models",
        ".csv": config.DATA_DIR / "timecourses", ".pdf": config.PAPERS_DIR}
for f in sorted(FILES.iterdir()):
    d = DEST.get(f.suffix.lower())
    if d and not (d / f.name).exists():
        d.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, d / f.name)
        print("added", d / f.name)
print("model files ready")

## One example of every capability (main text: Results; S1 File, Section S9)

47 requests, one run each in a clean session.

### M01: BioModels search

In [ ]:
new_chat()
ask('Search BioModels for models of bistability in the MAPK cascade and list the five best matches with their IDs.')

### M02: BioModels entry details and download

In [ ]:
new_chat()
ask('Download BIOMD0000000027 from BioModels and tell me which paper it comes from and what it models.')

### M03: Models of a paper

In [ ]:
new_chat()
ask('Which BioModels entries belong to the paper by Markevich, Hoek and Kholodenko (2004) on signaling switches in MAPK cascades?')

### M04: Proteins of a model (UniProt)

In [ ]:
new_chat()
ask('Which proteins does Markevich2004 represent? Give their UniProt entries.')

### M05: A model's ODEs

In [ ]:
new_chat()
ask('Show the ODEs of Markevich2004 with all parameter values and initial conditions.')

### M06: Typed equations to an SBML model

In [ ]:
new_chat()
ask('Turn these equations into an SBML model called cubic_fold: dx/dt = p + x - x^3, with p = 0 and x(0) = 1. Then show its ODEs.')

### M07: XPPAUT file text to an SBML model

In [ ]:
new_chat()
ask("Convert this XPPAUT model into SBML and call it toggle_xpp:\n# genetic toggle switch\nu' = a/(1+v^2) - u\nv' = a/(1+u^2) - v\npar a=3\ninit u=2, v=0.1\ndone")

### M08: Saving a changed copy of a model

In [ ]:
new_chat()
ask('Save a copy of Markevich2004 with MKP3 = 50 nM as Markevich2004_MKP3_50, and tell me what changed.')

### M09: SBML annotations

In [ ]:
new_chat()
ask('Show the annotations of Markevich2004: name, authors, notes and cross-references.')

### M10: Model classification

In [ ]:
new_chat()
ask('What is the dimension of Markevich2004 and which bifurcations are possible in it?')

In [ ]:
reply('yes')   # Dynalyst asked (parameter); reply yes

### M11: Model reduction (conservation laws, QSSA)

In [ ]:
new_chat()
ask('Reduce Markevich2004 with the quasi-steady-state approximation and tell me whether the reduced model can replace the full one.')

In [ ]:
reply('yes')   # Dynalyst asked (parameter); reply yes

### M12: Reaction network theory (CRNT)

In [ ]:
new_chat()
ask('Can the reaction network of Markevich2004 have several steady states? Use chemical reaction network theory.')

### M13: Drawing the reaction network

In [ ]:
new_chat()
ask('Draw the reaction network of Markevich2004.')

### M12b: Reaction network theory (CRNT), mass-action model

In [ ]:
new_chat()
ask('Can the reaction network of BIOMD0000000026 (the elementary-reaction version of the Markevich model) have several steady states? Use chemical reaction network theory.')

### M13b: Drawing the reaction network, mass-action model

In [ ]:
new_chat()
ask('Draw the reaction network of BIOMD0000000026.')